In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install stable-ts
!sudo apt update && sudo apt install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.1/189.1 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 32.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for stable-ts: filename=stable_ts-2.19.1-py3-none-any.whl size=162765 sha256=38b484181b103362f9f785ebfca4f8acca708b9efcfcfd40b67fd809bd211ef8
  Stored in directory: /root/.cache/pip/wheels/18/10/fe/b9c3ab284b29f6a379c58d57b7447055964d730f2f2ea7f6e4
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=4dcb31f6a23372063dbfccbf1937e583a1c696dd44a9a04bd92a4e47f4e9a840
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built stable-ts openai-whisper
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https

In [3]:
import stable_whisper
import os
import subprocess
import json
import re

model = stable_whisper.load_model('medium')
language = 'id'
work_dir = '/content/drive/MyDrive/Whisper'

def get_resolution(video_path):
    result = subprocess.run([
        'ffprobe', '-v', 'quiet', '-print_format', 'json',
        '-show_streams', video_path
    ], capture_output=True, text=True)
    streams = json.loads(result.stdout)['streams']
    video = next(s for s in streams if s['codec_type'] == 'video')
    return video['width'], video['height']

def make_ass_header(width, height, fontsize=28):
    margin_v = height // 6
    return f"""[Script Info]
ScriptType: v4.00+
PlayResX: {width}
PlayResY: {height}

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,{fontsize},&H00FFFFFF,&H00FFFFFF,&H00000000,&H00000000,0,0,0,0,100,100,0,0,1,1,0,2,10,10,{margin_v},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

def inject_karaoke_colors(events_body):
    lines = []
    for line in events_body.splitlines():
        if line.startswith('Dialogue:'):
            # Kata aktif = kuning, langsung reset ke putih setelahnya
            line = re.sub(
                r'\{(\\k\d+)\}([^{]+)',
                r'{\1\\1c&H0000FFFF&}\2{\\1c&H00FFFFFF&}',
                line
            )
        lines.append(line)
    return '\n'.join(lines)

if os.path.exists(f"{work_dir}/video"):
    os.makedirs(f"{work_dir}/caption", exist_ok=True)
    os.makedirs(f"{work_dir}/output", exist_ok=True)

    for index, source in enumerate(os.listdir(f"{work_dir}/video")):
        name = os.path.splitext(source)[0]
        path_input = f"{work_dir}/video/{source}"
        path_ass = f"{work_dir}/caption/{name}.ass"
        path_output = f"{work_dir}/output/{name}.mp4"

        if not os.path.exists(path_input):
            print(f"File tidak ditemukan: {path_input}")
            continue

        print(f"Memproses transkripsi {index+1}: {name}")

        w, h = get_resolution(path_input)

        result = model.transcribe(path_input, language=language)
        result.split_by_gap(0.5).split_by_length(max_words=3)
        result.to_ass(path_ass)

        with open(path_ass, 'r') as f:
            content = f.read()
        events_body = re.split(r'\[Events\].*?\n.*?\n', content, flags=re.DOTALL)[1]
        with open(path_ass, 'w') as f:
            f.write(make_ass_header(w, h) + inject_karaoke_colors(events_body))

        subprocess.run([
            'ffmpeg', '-i', path_input,
            '-vf', f"ass={path_ass}",
            '-c:a', 'copy', path_output, '-y'
        ], check=True)

        print(f"BERHASIL! Output: {path_output}")
else:
    print("Folder tidak ada.")

100%|██████████████████████████████████████| 1.42G/1.42G [00:13<00:00, 111MiB/s]


Memproses transkripsi 1: Lamborghini pernah jadi milik Indonesia？!


Transcribe: 100%|██████████| 127.87/127.87 [00:30<00:00,  4.23sec/s]


Saved: /content/drive/MyDrive/Whisper/srt/Lamborghini pernah jadi milik Indonesia？!.ass
BERHASIL! Output: /content/drive/MyDrive/Whisper/output/Lamborghini pernah jadi milik Indonesia？!.mp4
Memproses transkripsi 2: satwa ini cuma ada di Indonesia #podcast


Transcribe: 100%|██████████| 50.76/50.76 [00:11<00:00,  4.57sec/s]


Saved: /content/drive/MyDrive/Whisper/srt/satwa ini cuma ada di Indonesia #podcast.ass
BERHASIL! Output: /content/drive/MyDrive/Whisper/output/satwa ini cuma ada di Indonesia #podcast.mp4
